In [64]:
import pandas as pd
import numpy as np
from gensim.test.utils import datapath
from gensim import utils

#### Loading `tisch` and `sept` into sentences

In [65]:
sept = pd.read_pickle("./pickles/sept.pickle")
tisch = pd.read_pickle("./pickles/tisch.pickle")

In [66]:
sept_verses = list(sept.groupby(["book","chapter","verse"])['text'].apply(lambda words: " ".join(words).split()).values)
tisch_verses = list(tisch.groupby(["book","chapter","verse"])['text'].apply(lambda words: " ".join(words).split()).values)

In [67]:
print("sept verses", sept_verses[:5])
print("tisch verses", tisch_verses[:5])

sept verses [['εν', 'αρχη', 'εποιησεν', 'ο', 'θεος', 'τον', 'ουρανον', 'και', 'την', 'γην'], ['η', 'δε', 'γη', 'ην', 'αορατος', 'και', 'ακατασκευαστος', 'και', 'σκοτος', 'επανω', 'της', 'αβυσσου', 'και', 'πνευμα', 'θεου', 'επεφερετο', 'επανω', 'του', 'υδατος'], ['και', 'ειπεν', 'ο', 'θεος', 'γενηθητω', 'φως', 'και', 'εγενετο', 'φως'], ['και', 'ειδεν', 'ο', 'θεος', 'το', 'φως', 'οτι', 'καλον', 'και', 'διεχωρισεν', 'ο', 'θεος', 'ανα', 'μεσον', 'του', 'φωτος', 'και', 'ανα', 'μεσον', 'του', 'σκοτους'], ['και', 'εκαλεσεν', 'ο', 'θεος', 'το', 'φως', 'ημεραν', 'και', 'το', 'σκοτος', 'εκαλεσεν', 'νυκτα', 'και', 'εγενετο', 'εσπερα', 'και', 'εγενετο', 'πρωι', 'ημερα', 'μια']]
tisch verses [['βίβλος', 'γενέσεως', 'ἰησοῦ', 'χριστοῦ', 'υἱοῦ', 'δαυεὶδ', 'υἱοῦ', 'ἀβραάμ'], ['ἀβραὰμ', 'ἐγέννησεν', 'τὸν', 'ἰσαάκ', 'ἰσαὰκ', 'δὲ', 'ἐγέννησεν', 'τὸν', 'ἰακώβ', 'ἰακὼβ', 'δὲ', 'ἐγέννησεν', 'τὸν', 'ἰούδαν', 'καὶ', 'τοὺς', 'ἀδελφοὺς', 'αὐτοῦ'], ['ἰούδας', 'δὲ', 'ἐγέννησεν', 'τὸν', 'φάρες', 'καὶ', 'τὸν', 'ζάρα

#### Book, verse, chapter tisch and sept db

In [68]:
sept_plain = sept.drop(['rmac','str'], axis="columns")
tisch_plain = tisch.drop(['rmac','str'], axis="columns")
print("Sept, then tisch.")
display(sept_plain.head(2))
display(tisch_plain.head(2))

Sept, then tisch.


,text,verse,chapter,book
0,εν,1,1,1
1,αρχη,1,1,1


,text,verse,chapter,book
0,βίβλος,1,1,40
1,γενέσεως,1,1,40


## Making the word2vec Model

In [69]:
import gensim.models
model = gensim.models.Word2Vec(
    sentences= sept_verses + tisch_verses,
    min_count=1,
)

In [ ]:
model = gensim.models.Word2Vec(vector_size = 100, window = 5, min_count = 1, workers = 4)
model.build_vocab(sept_verses + tisch_verses)
model.train(
    sept_verses + tisch_verses, 
    total_examples = model.corpus_count,
    epochs = 100,
    compute_loss = True
)

(49208920, 61292500)

In [142]:
model.wv.most_similar("θεος")[:3]

[('κυριος', 0.6632614731788635),
 ('πατηρ', 0.5930590629577637),
 ('βασιλευς', 0.5268449187278748)]

In [143]:
print(model.wv.index_to_key[:4])

['και', 'εν', 'καὶ', 'του']


In [74]:
matt_opening = tisch.query("book == 40")[0:5]
luke_opening = tisch.query("book == 41")[0:5]
mark_opening = tisch.query("book == 42")[0:5]

### Making the Sentence comparision method

In [75]:
np.linalg.norm(model.wv['καὶ'] - model.wv['τοῦ'])

26.761635

In [76]:
model.wv.similarity('καὶ', 'τοῦ')

0.6830681

In [77]:
def euclidean_distance(word1, word2):
    return np.linalg.norm(model.wv[word1] - model.wv[word2])

In [ ]:
def bigramify(frame1: pd.DataFrame, frame2: pd.DataFrame):
    assert len(frame1) == len(frame2), f"len(frame1) = {len(frame1)}, len(frame2) = {len(frame2)} -> must be equal"
    
    frame1 = frame1[frame1['rmac'].str.startswith(('v','n', 'a'))]
    frame2 = frame2[frame2['rmac'].str.startswith(('v','n', 'a'))]
    

    min_distances = [
        min([
            (word1, word2, euclidean_distance(word1, word2))
            for word2 in frame2['text']
        ])
        for word1 in frame1['text']
    ]
    return(min_distances)


In [ ]:
def euclidean_distance(word1, word2):
    return np.linalg.norm(model.wv[word1] - model.wv[word2])

def find_distance(frame1: pd.DataFrame, frame2: pd.DataFrame, filter_junk_words: bool = True) -> float:
    """Returns the distance in semantic meaning of two dataframes of equal length"""

    assert len(frame1) == len(frame2), f"len(frame1) = {len(frame1)}, len(frame2) = {len(frame2)} -> must be equal"
    
    # Removes any word that is not a verb, noun, adjective
    if filter_junk_words:
        important_structures = {"Adjective":"a", "Adverb":"d", "Numeral":"m", "Verb":"v", "Preposition":"p", "Noun":"n", }
        frame1 = frame1[frame1['rmac'].str.startswith(important_structures.values)]
        frame2 = frame2[frame2['rmac'].str.startswith(important_strctures.values)]
    

    
    minimum_distances = [
        min(
            ((word1, word2, euclidean_distance(word1, word2)) for word2 in frame2['text']),
            key=lambda x: x[2]
        )
        for word1 in frame1['text']
    ]

    average = sum(item[2] for item in minimum_distances) / len(minimum_distances)
    return average

find_distance(luke_opening, matt_opening, False)

8.889607429504395